In [1]:
# ===================================================================
# PHASE 1: SETUP & IMPORTS
# ===================================================================
# Install required libraries
!pip install efficientnet-pytorch -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from efficientnet_pytorch import EfficientNet
import torchvision.transforms as transforms

In [9]:
 #===================================================================
# PHASE 2: CONFIGURATION
# ===================================================================

# --- IMPORTANT: UPDATE THESE TWO PATHS ---
GRU_MODEL_PATH = "/kaggle/input/my-riot-detection-model/best_gru_model.pth"
VIDEO_TO_TEST_PATH = "/kaggle/input/dataset/Normal-Videos-Part-1/Normal_Videos_006_x264.mp4"

In [4]:
# --- Model & Data Parameters (must match your training script) ---
EFFICIENTNET_MODEL_NAME = 'efficientnet-b0'
FIXED_SEQUENCE_LENGTH = 100 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_FEATURES = 1280
HIDDEN_SIZE = 256
NUM_LAYERS = 2
NUM_CLASSES = 2
DROPOUT = 0.3
CLASS_NAMES = ['Normal', 'Anomaly']

print(f"Using device: {DEVICE}")


Using device: cuda


In [5]:
# ===================================================================
# PHASE 3: RELOAD ALL NECESSARY FUNCTIONS AND CLASSES
# ===================================================================

# --- Helper Functions for Video Processing ---
def extract_frames(video_path, frame_rate=1):
    frames = []
    try:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps == 0: return np.array([])
        frame_interval = int(fps / frame_rate) if fps >= frame_rate else 1
        count = 0
        success, frame = cap.read()
        while success:
            if count % frame_interval == 0:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (224, 224))
                frames.append(frame)
            success, frame = cap.read()
            count += 1
    finally:
        if 'cap' in locals() and cap.isOpened():
            cap.release()
    return np.array(frames)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_features(frames, feature_extractor, batch_size=32):
    features = []
    with torch.no_grad():
        for i in range(0, len(frames), batch_size):
            batch = frames[i:i+batch_size]
            batch = torch.stack([transform(f).to(DEVICE) for f in batch])
            output = feature_extractor(batch)
            features.append(output.cpu().numpy())
    return np.vstack(features)

# --- GRU Model Class Definition (must be identical to training script) ---
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.linear = nn.Linear(hidden_size * 2, output_size)
    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_time_step_out = gru_out[:, -1, :]
        out = self.linear(last_time_step_out)
        return out


In [10]:
# ===================================================================
# PHASE 4: MAIN INFERENCE LOGIC
# ===================================================================

if __name__ == "__main__":
    # --- 1. Load Models ---
    print("Loading models...")
    # Load EfficientNet feature extractor
    efficientnet = EfficientNet.from_pretrained(EFFICIENTNET_MODEL_NAME)
    efficientnet._fc = nn.Identity()
    efficientnet = efficientnet.to(DEVICE)
    efficientnet.eval()
    
    # Load trained GRU model
    gru_model = GRUModel(INPUT_FEATURES, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT).to(DEVICE)
    gru_model.load_state_dict(torch.load(GRU_MODEL_PATH, map_location=DEVICE))
    gru_model.eval()
    print("Models loaded successfully.")

    # --- 2. Process the Input Video ---
    print(f"\nProcessing video: {VIDEO_TO_TEST_PATH}")
    frames = extract_frames(VIDEO_TO_TEST_PATH)
    if len(frames) == 0:
        print("Could not extract frames from the video. Please check the file path and format.")
    else:
        features = get_features(frames, efficientnet)
        
        # --- 3. Prepare Sequence for GRU Model ---
        # Pad or truncate the sequence
        if features.shape[0] > FIXED_SEQUENCE_LENGTH:
            features = features[:FIXED_SEQUENCE_LENGTH]
        else:
            padding = np.zeros((FIXED_SEQUENCE_LENGTH - features.shape[0], features.shape[1]))
            features = np.vstack((features, padding))
        
        # --- 4. Get Prediction ---
        with torch.no_grad():
            # Convert to tensor, add batch dimension, and move to device
            sequence_tensor = torch.tensor(features, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            
            # Get model output (logits)
            output = gru_model(sequence_tensor)
            
            # Convert logits to probabilities using softmax
            probabilities = torch.softmax(output, dim=1)
            
            # Get the top prediction
            confidence, predicted_class_idx = torch.max(probabilities, 1)
            
            predicted_class_name = CLASS_NAMES[predicted_class_idx.item()]
            confidence_score = confidence.item()

        # --- 5. Display Result ---
        print("\n--- PREDICTION ---")
        print(f"Predicted Class: {predicted_class_name}")
        print(f"Confidence: {confidence_score:.2%}")
        print("--------------------")

Loading models...
Loaded pretrained weights for efficientnet-b0
Models loaded successfully.

Processing video: /kaggle/input/dataset/Normal-Videos-Part-1/Normal_Videos_006_x264.mp4

--- PREDICTION ---
Predicted Class: Normal
Confidence: 96.07%
--------------------
